# Reranking: Improving Retrieval Quality

## Overview

Initial retrieval (e.g., FAISS similarity search) returns documents based on embedding distance — fast but not always accurate. **Reranking** takes these initial results and re-scores them with a more sophisticated model to get truly relevant documents to the top.

We demonstrate two reranking methods:

| Method | How it scores | Speed | Accuracy |
|---|---|---|---|
| **LLM-based** | Ask the LLM to rate each document 1-10 | Slow (one LLM call per doc) | High (leverages LLM reasoning) |
| **Cross-Encoder** | Feed (query, doc) pairs into a specialized model | Fast | High (trained specifically for relevance) |

## Models Used

- **Embeddings**: `mxbai-embed-large:335m` via Ollama (initial retrieval)
- **LLM**: `gemma3:12b` via Ollama (Method 1 reranking + answer generation)
- **Cross-Encoder**: `cross-encoder/ms-marco-MiniLM-L-6-v2` (Method 2 reranking)

<div style="text-align: center;">

<img src="./images/reranking-visualization.svg" alt="Reranking" style="width:100%; height:auto;">
</div>

<div style="text-align: center;">

<img src="./images/reranking_comparison.svg" alt="Reranking Comparison" style="width:100%; height:auto;">
</div>

---
## Step 0: Import Packages

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from sentence_transformers import CrossEncoder

---
## Step 1: Set Up Models

In [2]:
embedding_model = OllamaEmbeddings(model="mxbai-embed-large:335m")
llm = ChatOllama(temperature=0, model="gemma3:12b", max_tokens=4000)

print("Embedding model and LLM ready")

Embedding model and LLM ready


---
## Step 2: Load PDF, Chunk, Build Vector Store

In [3]:
path = "data/Understanding_Climate_Change.pdf"

loader = PyPDFLoader(path)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, length_function=len
)
chunks = text_splitter.split_documents(documents)

for chunk in chunks:
    chunk.page_content = chunk.page_content.replace("\t", " ")

vectorstore = FAISS.from_documents(chunks, embedding_model)

print(f"Loaded {len(documents)} pages, split into {len(chunks)} chunks")
print(f"Vector store created")

Loaded 33 pages, split into 97 chunks
Vector store created


---
---
# Method 1: LLM-Based Reranking

Ask the LLM to score each document's relevance to the query on a 1-10 scale, then sort by score.

<div style="text-align: center;">
<img src="./images/rerank_llm.svg" alt="LLM Reranking" style="width:40%; height:auto;">
</div>

---
## Step 3: Retrieve Initial Documents

In [4]:
query = "What are the impacts of climate change on biodiversity?"
print(f"Query: {query}\n")

initial_docs = vectorstore.similarity_search(query, k=15)

print(f"Retrieved {len(initial_docs)} initial documents")
print("\nTop 3 from initial retrieval:")
for i, doc in enumerate(initial_docs[:3]):
    print(f"  Doc {i+1}: {doc.page_content[:150]}...")

Query: What are the impacts of climate change on biodiversity?

Retrieved 15 initial documents

Top 3 from initial retrieval:
  Doc 1: Climate change is altering terrestrial ecosystems by shifting habitat ranges, changing species 
distributions, and impacting ecosystem functions. Fore...
  Doc 2: cultural perceptions. 
Youth Engagement 
Youth are vital stakeholders in climate action. Empowering young people through education, 
activism, and lea...
  Doc 3: protection, and habitat creation. 
Climate-Resilient Conservation 
Conservation strategies must account for climate change impacts to be effective. Th...


---
## Step 4: Rerank with LLM

For each document, we ask the LLM: *"On a scale of 1-10, how relevant is this document to the query?"* Then we sort by score and keep the top 3.

In [5]:
rerank_schema = {
    "title": "RatingScore",
    "type": "object",
    "properties": {
        "relevance_score": {
            "type": "number",
            "description": "The relevance score of a document to a query, from 1 to 10"
        }
    },
    "required": ["relevance_score"]
}

rerank_prompt = PromptTemplate(
    input_variables=["query", "doc"],
    template=(
        "On a scale of 1-10, rate the relevance of the following document to the query. "
        "Consider the specific context and intent of the query, not just keyword matches.\n"
        "Query: {query}\nDocument: {doc}\nRelevance Score:"
    )
)

rerank_chain = rerank_prompt | llm.with_structured_output(rerank_schema)

# Score each document
scored_docs = []
for i, doc in enumerate(initial_docs):
    result = rerank_chain.invoke({"query": query, "doc": doc.page_content})
    score = float(result["relevance_score"])
    scored_docs.append((doc, score))
    print(f"  Doc {i+1}: score = {score:.1f}  |  {doc.page_content[:80]}...")

# Sort by score descending, keep top 3
scored_docs.sort(key=lambda x: x[1], reverse=True)
top_n = 3
reranked_docs_llm = [doc for doc, _ in scored_docs[:top_n]]

print(f"\nTop {top_n} after LLM reranking:")
for i, doc in enumerate(reranked_docs_llm):
    print(f"  Doc {i+1}: {doc.page_content[:150]}...")

  Doc 1: score = 9.0  |  Climate change is altering terrestrial ecosystems by shifting habitat ranges, ch...
  Doc 2: score = 6.0  |  cultural perceptions. 
Youth Engagement 
Youth are vital stakeholders in climate...
  Doc 3: score = 8.0  |  protection, and habitat creation. 
Climate-Resilient Conservation 
Conservation ...
  Doc 4: score = 3.0  |  goals. Policies should promote synergies between biodiversity conservation and c...
  Doc 5: score = 6.5  |  rehabilitation. Engaging local communities in restoration projects ensures susta...
  Doc 6: score = 8.0  |  development of eco-friendly fertilizers and farming techniques is essential for ...
  Doc 7: score = 4.0  |  Local communities are often on the front lines of climate impacts and can be pow...
  Doc 8: score = 8.0  |  Freshwater Ecosystems 
Freshwater ecosystems, including rivers, lakes, and wetla...
  Doc 9: score = 6.5  |  managed retreats. 
Extreme Weather Events 
Climate change is linked to an increa...
  Doc 10: score = 8

---
## Step 5: Generate Answer from LLM-Reranked Documents

In [6]:
context_llm = "\n\n".join(doc.page_content for doc in reranked_docs_llm)

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an assistant for question-answering tasks. Use the following context to answer the question."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

answer_chain = answer_prompt | llm | StrOutputParser()
answer_llm = answer_chain.invoke({"context": context_llm, "question": query})

print(f"Question: {query}\n")
print(f"Answer (LLM reranking): {answer_llm}")

Question: What are the impacts of climate change on biodiversity?

Answer (LLM reranking): According to the text, climate change impacts biodiversity in the following ways:

*   **Terrestrial Ecosystems:** Climate change is shifting habitat ranges, changing species distributions, and impacting ecosystem functions, leading to a loss of biodiversity and disruption of ecological balance.
*   **Marine Ecosystems:** Rising sea temperatures, ocean acidification, and changing currents affect marine biodiversity, from coral reefs to deep-sea habitats. Species migration and changes in reproductive cycles can disrupt marine food webs and fisheries.


---
## Step 6: Why Reranking Matters — A Quick Demo

Embedding-based retrieval matches surface-level similarity. It might rank "The capital of France is great" above a document that actually **answers** the question. Reranking fixes this.

In [7]:
demo_chunks = [
    "The capital of France is great.",
    "The capital of France is huge.",
    "The capital of France is beautiful.",
    "Have you ever visited Paris? It is a beautiful city where you can eat delicious food and see the Eiffel Tower. "
    "I really enjoyed all the cities in France, but its capital with the Eiffel Tower is my favorite city.",
    "I really enjoyed my trip to Paris, France. The city is beautiful and the food is delicious. "
    "I would love to visit again. Such a great capital city.",
]
demo_docs = [Document(page_content=text) for text in demo_chunks]
demo_vs = FAISS.from_documents(demo_docs, embedding_model)

demo_query = "what is the capital of france?"
print(f"Query: {demo_query}\n")

# Baseline: embedding similarity
print("--- Baseline (embedding similarity) ---")
baseline = demo_vs.similarity_search(demo_query, k=2)
for i, doc in enumerate(baseline):
    print(f"  {i+1}) {doc.page_content[:100]}")

# Reranked with LLM
print("\n--- After LLM reranking ---")
demo_initial = demo_vs.similarity_search(demo_query, k=5)
demo_scored = []
for doc in demo_initial:
    result = rerank_chain.invoke({"query": demo_query, "doc": doc.page_content})
    demo_scored.append((doc, float(result["relevance_score"])))
demo_scored.sort(key=lambda x: x[1], reverse=True)
for i, (doc, score) in enumerate(demo_scored[:2]):
    print(f"  {i+1}) [score={score:.1f}] {doc.page_content[:100]}")

Query: what is the capital of france?

--- Baseline (embedding similarity) ---
  1) The capital of France is great.
  2) The capital of France is huge.

--- After LLM reranking ---
  1) [score=9.0] The capital of France is great.
  2) [score=9.0] The capital of France is huge.


---
---
# Method 2: Cross-Encoder Reranking

Instead of using the LLM (slow, one call per document), we use a **Cross-Encoder** — a small model specifically trained to score (query, document) relevance. Much faster.

<div style="text-align: center;">
<img src="./images/rerank_cross_encoder.svg" alt="Cross-Encoder Reranking" style="width:40%; height:auto;">
</div>

---
## Step 7: Load the Cross-Encoder Model

In [8]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Cross-Encoder loaded")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cross-Encoder loaded


---
## Step 8: Retrieve, Rerank with Cross-Encoder, Display Results

In [9]:
query_ce = "What are the impacts of climate change on biodiversity?"
print(f"Query: {query_ce}\n")

# Initial retrieval
initial_k = 10
rerank_top_k = 5
initial_docs_ce = vectorstore.similarity_search(query_ce, k=initial_k)
print(f"Retrieved {len(initial_docs_ce)} initial documents")

# Score each (query, document) pair with the Cross-Encoder
pairs = [[query_ce, doc.page_content] for doc in initial_docs_ce]
ce_scores = cross_encoder.predict(pairs)

print("\nCross-Encoder scores:")
for i, (doc, score) in enumerate(zip(initial_docs_ce, ce_scores)):
    print(f"  Doc {i+1}: score = {score:.4f}  |  {doc.page_content[:80]}...")

# Sort by score descending, keep top-k
scored_pairs = sorted(zip(ce_scores, initial_docs_ce), key=lambda x: x[0], reverse=True)
reranked_docs_ce = [doc for _, doc in scored_pairs[:rerank_top_k]]

print(f"\nTop {rerank_top_k} after Cross-Encoder reranking:")
for i, doc in enumerate(reranked_docs_ce):
    print(f"  Doc {i+1}: {doc.page_content[:150]}...")

Query: What are the impacts of climate change on biodiversity?

Retrieved 10 initial documents

Cross-Encoder scores:
  Doc 1: score = 7.5049  |  Climate change is altering terrestrial ecosystems by shifting habitat ranges, ch...
  Doc 2: score = 4.1821  |  cultural perceptions. 
Youth Engagement 
Youth are vital stakeholders in climate...
  Doc 3: score = 5.7396  |  protection, and habitat creation. 
Climate-Resilient Conservation 
Conservation ...
  Doc 4: score = 5.5052  |  goals. Policies should promote synergies between biodiversity conservation and c...
  Doc 5: score = 1.0766  |  rehabilitation. Engaging local communities in restoration projects ensures susta...
  Doc 6: score = 2.5348  |  development of eco-friendly fertilizers and farming techniques is essential for ...
  Doc 7: score = 4.2930  |  Local communities are often on the front lines of climate impacts and can be pow...
  Doc 8: score = -1.7215  |  Freshwater Ecosystems 
Freshwater ecosystems, including rivers, lakes

---
## Step 9: Generate Answer from Cross-Encoder-Reranked Documents

In [10]:
context_ce = "\n\n".join(doc.page_content for doc in reranked_docs_ce)

answer_ce = answer_chain.invoke({"context": context_ce, "question": query_ce})

print(f"Question: {query_ce}\n")
print(f"Answer (Cross-Encoder reranking): {answer_ce}")

Question: What are the impacts of climate change on biodiversity?

Answer (Cross-Encoder reranking): According to the text, climate change is impacting biodiversity in the following ways:

*   **Terrestrial Ecosystems:** Climate change is altering terrestrial ecosystems by shifting habitat ranges, changing species distributions, and impacting ecosystem functions. Forests, grasslands, and deserts are experiencing these shifts.
*   **Marine Ecosystems:** Rising sea temperatures, ocean acidification, and changing currents affect marine biodiversity, from coral reefs to deep-sea habitats. Species migration and changes in reproductive cycles can disrupt marine food webs and fisheries.


---
## Summary

| Step | What happened |
|---|---|
| 1-2 | Set up models, loaded PDF, built vector store |
| **Method 1: LLM Reranking** | |
| 3 | Retrieved 15 initial documents |
| 4 | LLM scored each document 1-10, sorted, kept top 3 |
| 5 | Generated answer from reranked documents |
| 6 | Demo: showed how reranking fixes embedding-only retrieval |
| **Method 2: Cross-Encoder** | |
| 7 | Loaded Cross-Encoder model |
| 8 | Scored all (query, doc) pairs in one batch, sorted, kept top 5 |
| 9 | Generated answer from reranked documents |

**Key insight:** Initial embedding retrieval casts a wide net. Reranking is the precision filter that ensures the most relevant documents bubble to the top. LLM reranking is more flexible but slower; Cross-Encoder reranking is faster and purpose-built for relevance scoring. Both significantly improve answer quality over baseline retrieval.